# 📡 Telecom Voice of Customer (VoC) Intelligence Suite
### Multilingual Sentiment Classification: ML, DL & Ensembles on Google Colab (GPU Accelerated)

This notebook provides a complete environment to train, benchmark, and deploy models on customer feedback across Bangladesh's premier telecom operators (**Grameenphone**, **Banglalink**, and **Robi Axiata**).

#### Models Evaluated:
1. **Ensemble Machine Learning (Soft-Voting Classifier)**: Balanced Logistic Regression + Calibrated LinearSVC + ComplementNB over Character N-Gram TF-IDF.
2. **BiLSTM with Self-Attention & Char-CNN**: Dual word + character-level convolutions tailored for phonetic Romanized Banglish (`faltu`, `onek baje`), native Bengali script (`অনেক ভালো`), and English.
3. **Multilingual Transformer (`paraphrase-multilingual-MiniLM-L12-v2`)**: Pretrained dense multilingual representations (384-dim) with a regularized neural classification head.
4. **BUET BanglaBERT (`csebuetnlp/banglabert`)**: SOTA Bengali language model pretrained by BUET on 110 GB of native Bengali text (768-dim).

> 💡 **Hardware Tip:** In Google Colab, go to `Runtime` ➔ `Change runtime type` ➔ select **T4 GPU** for high-speed training.

In [ ]:
# 1. Verify GPU Acceleration
import torch

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Running on CPU. Recommend enabling GPU via Runtime > Change runtime type > T4 GPU.")

In [ ]:
# 2. Clone Repository & Setup Environment
import os
from pathlib import Path

if not Path("data/processed").exists():
    !git clone https://github.com/rh61632/telecom-Voice-of-Customer_intelligence.git
    %cd telecom-Voice-of-Customer_intelligence

!pip install -q transformers sentence-transformers scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# 3. Load & Preprocess All 3 Operator Datasets
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

LABEL_MAP = {"Negative": 0, "Neutral": 1, "Positive": 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

files = {
    "Grameenphone": "data/processed/mygp_classified_reviews.csv",
    "Banglalink": "data/processed/mybl_classified_reviews.csv",
    "Robi": "data/processed/myrobi_classified_reviews.csv",
}

dfs = []
for op, p in files.items():
    d = pd.read_csv(p)
    if "operator" not in d.columns:
        d["operator"] = op
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
df = df.dropna(subset=["review_text", "sentiment"]).copy()
df["review_text"] = df["review_text"].astype(str).str.strip()
df = df[df["review_text"].str.len() > 0]
df = df[df["sentiment"].isin(LABEL_MAP.keys())].reset_index(drop=True)
df["label"] = df["sentiment"].map(LABEL_MAP)

# --- SHORT TEXT FILTERING TOGGLE ---
# Set MIN_WORDS = 1 to keep all reviews (N = 4,500)
# Set MIN_WORDS = 3 to exclude very small 1-2 word messages (N = 3,886)
# Set MIN_WORDS = 5 to keep only detailed feedback (N = 2,285)
MIN_WORDS = 1  # <-- Change this value to experiment with short-text filtering!
df["word_count"] = df["review_text"].str.split().apply(len)
if MIN_WORDS > 1:
    print(f"Applying filter: Keeping reviews with >= {MIN_WORDS} words...")
    df = df[df["word_count"] >= MIN_WORDS].reset_index(drop=True)
    print(f"Retained {len(df)} substantive reviews.")

def clean_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()

df["clean_text"] = df["review_text"].apply(clean_text)

print(f"Successfully loaded {len(df)} total reviews across 3 operators.")
print("Class Distribution:")
for s, count in df["sentiment"].value_counts().items():
    print(f"  - {s:8s}: {count:5d} ({count / len(df) * 100:.2f}%)")


--- 
## 🏆 Model 1: Soft-Voting Ensemble Machine Learning
Combines three distinct linear and probabilistic classifiers trained over sub-word Character N-Gram TF-IDF:
1. **Balanced Logistic Regression**: Convex linear boundaries penalizing minority class errors.
2. **Calibrated LinearSVC**: Maximum-margin linear separation with Sigmoid probability calibration.
3. **Complement Naive Bayes**: Probabilistic classifier specifically designed for severe text class skew.

In [ ]:
# 4. Train & Evaluate Soft-Voting Ensemble ML (5-Fold CV)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

tfidf_extractor = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=3,
    sublinear_tf=True
)

clf_lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
clf_svc = CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=42))
clf_cnb = ComplementNB(norm=True)

ensemble_pipeline = Pipeline([
    ("tfidf", tfidf_extractor),
    ("voting", VotingClassifier(
        estimators=[("lr", clf_lr), ("svc", clf_svc), ("cnb", clf_cnb)],
        voting="soft",
        weights=[2, 2, 1]
    ))
])

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ensemble_scores = cross_validate(
    ensemble_pipeline,
    df["review_text"],
    df["sentiment"],
    cv=cv_strat,
    scoring={"f1_macro": "f1_macro", "acc": "accuracy"},
    n_jobs=-1
)

ensemble_f1 = float(np.mean(ensemble_scores["test_f1_macro"]))
ensemble_acc = float(np.mean(ensemble_scores["test_acc"]))
ensemble_folds_f1 = [float(s) for s in ensemble_scores["test_f1_macro"]]
ensemble_folds_acc = [float(s) for s in ensemble_scores["test_acc"]]

print(f"===> Ensemble ML 5-Fold Macro-F1: {ensemble_f1:.4f} (±{np.std(ensemble_folds_f1):.4f})")
print(f"===> Ensemble ML 5-Fold Accuracy: {ensemble_acc * 100:.2f}%")
print(f"Fold Scores: {[round(s, 4) for s in ensemble_folds_f1]}")

--- 
## 🧠 Model 2: BiLSTM with Self-Attention & Char-CNN
Combines word embeddings, character-level 1D-CNN for Banglish phonetic roots, 2-layer BiLSTM, and additive attention pooling.

In [ ]:
# 5. Architecture Definition: BiLSTM with Self-Attention & Char-CNN
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class DualVocabulary:
    def __init__(self, max_words=10000, min_word_freq=2, max_chars=300):
        self.max_words = max_words
        self.min_word_freq = min_word_freq
        self.max_chars = max_chars
        self.word2idx = {"<pad>": 0, "<unk>": 1}
        self.char2idx = {"<pad>": 0, "<unk>": 1}
        self.idx2word = {0: "<pad>", 1: "<unk>"}

    def fit(self, texts):
        w_counts, c_counts = {}, {}
        for t in texts:
            for w in t.split():
                w_counts[w] = w_counts.get(w, 0) + 1
            for c in t:
                c_counts[c] = c_counts.get(c, 0) + 1
        
        sorted_words = sorted([w for w, cnt in w_counts.items() if cnt >= self.min_word_freq], key=lambda w: w_counts[w], reverse=True)[:self.max_words]
        for w in sorted_words:
            if w not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[w] = idx
                self.idx2word[idx] = w
                
        sorted_chars = sorted(c_counts.keys(), key=lambda c: c_counts[c], reverse=True)[:self.max_chars]
        for c in sorted_chars:
            if c not in self.char2idx:
                self.char2idx[c] = len(self.char2idx)

    def encode(self, text, max_words=64, max_word_chars=16):
        words = text.split()[:max_words]
        w_ids = [self.word2idx.get(w, 1) for w in words]
        c_matrix = []
        for w in words:
            c_ids = [self.char2idx.get(c, 1) for c in w[:max_word_chars]]
            c_ids += [0] * (max_word_chars - len(c_ids))
            c_matrix.append(c_ids)
        pad_len = max_words - len(w_ids)
        w_ids += [0] * pad_len
        c_matrix += [[0] * max_word_chars for _ in range(pad_len)]
        return w_ids, c_matrix

class BiLSTMAttentionDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.labels = labels
        self.w_data, self.c_data = [], []
        for t in texts:
            w, c = vocab.encode(t)
            self.w_data.append(w)
            self.c_data.append(c)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return torch.tensor(self.w_data[i], dtype=torch.long), torch.tensor(self.c_data[i], dtype=torch.long), torch.tensor(self.labels[i], dtype=torch.long)

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.query = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, h, mask=None):
        scores = self.query(torch.tanh(self.proj(h))).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), h).squeeze(1)
        return context, weights

class BiLSTMWithAttention(nn.Module):
    def __init__(self, word_vocab_size, char_vocab_size, word_dim=128, char_dim=32, char_filters=64, lstm_dim=128, num_classes=3, dropout=0.3):
        super().__init__()
        self.word_emb = nn.Embedding(word_vocab_size, word_dim, padding_idx=0)
        self.char_emb = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_conv = nn.Conv1d(char_dim, char_filters, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(word_dim + char_filters, lstm_dim, num_layers=2, bidirectional=True, batch_first=True, dropout=dropout)
        self.attention = Attention(lstm_dim * 2)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, w_ids, c_ids):
        b, seq, w_len = c_ids.shape
        w_vec = self.word_emb(w_ids)
        c_flat = c_ids.view(b * seq, w_len)
        c_conv = F.relu(self.char_conv(self.char_emb(c_flat).transpose(1, 2)))
        c_pooled, _ = torch.max(c_conv, dim=-1)
        c_vec = c_pooled.view(b, seq, -1)
        x = self.dropout(torch.cat([w_vec, c_vec], dim=-1))
        lstm_out, _ = self.bilstm(x)
        context, _ = self.attention(lstm_out, mask=(w_ids != 0))
        return self.classifier(context)

In [ ]:
# 6. Train & Evaluate BiLSTM on GPU (5-Fold Stratified CV)
from sklearn.metrics import f1_score, accuracy_score

def train_bilstm_cv(df, epochs=8, batch_size=32):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_f1, fold_acc = [], []
    
    class_counts = df["label"].value_counts().sort_index().values
    weights = torch.tensor([len(df) / (len(class_counts) * c) for c in class_counts], dtype=torch.float32).to(device)

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label"]), 1):
        train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]
        vocab = DualVocabulary()
        vocab.fit(train_df["clean_text"].tolist())
        
        train_ds = BiLSTMAttentionDataset(train_df["clean_text"].tolist(), train_df["label"].tolist(), vocab)
        val_ds = BiLSTMAttentionDataset(val_df["clean_text"].tolist(), val_df["label"].tolist(), vocab)
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        
        model = BiLSTMWithAttention(len(vocab.word2idx), len(vocab.char2idx)).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss(weight=weights)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

        best_f1, best_acc = 0.0, 0.0
        for ep in range(1, epochs + 1):
            model.train()
            for w_ids, c_ids, lbls in train_loader:
                w_ids, c_ids, lbls = w_ids.to(device), c_ids.to(device), lbls.to(device)
                optimizer.zero_grad()
                out = model(w_ids, c_ids)
                loss = criterion(out, lbls)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                
            model.eval()
            preds, targets = [], []
            with torch.no_grad():
                for w_ids, c_ids, lbls in val_loader:
                    w_ids, c_ids = w_ids.to(device), c_ids.to(device)
                    p = model(w_ids, c_ids).argmax(-1)
                    preds.extend(p.cpu().numpy())
                    targets.extend(lbls.numpy())
            
            val_f1 = f1_score(targets, preds, average="macro")
            val_acc = accuracy_score(targets, preds)
            scheduler.step(val_f1)
            if val_f1 > best_f1:
                best_f1, best_acc = val_f1, val_acc
                
        print(f"Fold {fold}/5 -> Macro-F1: {best_f1:.4f} | Accuracy: {best_acc * 100:.2f}%")
        fold_f1.append(best_f1)
        fold_acc.append(best_acc)
        
    print(f"\n===> BiLSTM Mean Macro-F1 : {np.mean(fold_f1):.4f} (±{np.std(fold_f1):.4f})")
    print(f"===> BiLSTM Mean Accuracy : {np.mean(fold_acc) * 100:.2f}%")
    return np.mean(fold_f1), np.mean(fold_acc), fold_f1, fold_acc

bilstm_f1, bilstm_acc, bilstm_folds_f1, bilstm_folds_acc = train_bilstm_cv(df)

--- 
## 🚀 Model 3: Multilingual Pretrained Transformer (MiniLM)
Uses `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384-dimensional dense multilingual embeddings).

In [ ]:
# 7. Extract Multilingual Transformer Embeddings on GPU
from transformers import AutoTokenizer, AutoModel

TRANSFORMER_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Loading Transformer: {TRANSFORMER_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_ID)
transformer_model = AutoModel.from_pretrained(TRANSFORMER_ID).to(device)
transformer_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

all_embeddings = []
batch_size = 64
texts = df["review_text"].tolist()
print(f"Extracting embeddings for {len(texts)} reviews on {device}...")

with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=96, return_tensors="pt").to(device)
        out = transformer_model(**enc)
        pooled = mean_pooling(out, enc["attention_mask"])
        normed = F.normalize(pooled, p=2, dim=1)
        all_embeddings.append(normed.cpu().numpy())

embeddings_matrix = np.vstack(all_embeddings)
print(f"Extracted Embedding Matrix: {embeddings_matrix.shape}")

In [ ]:
# 8. Train & Evaluate Multilingual Transformer Head (5-Fold CV)
class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class TransformerClassifier(nn.Module):
    def __init__(self, in_features=384, hidden=128, num_classes=3, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

def train_transformer_cv(embeddings, labels, epochs=15, batch_size=32):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_f1, fold_acc = [], []
    
    class_counts = pd.Series(labels).value_counts().sort_index().values
    weights = torch.tensor([len(labels) / (len(class_counts) * c) for c in class_counts], dtype=torch.float32).to(device)

    for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, labels), 1):
        train_ds = EmbeddingDataset(embeddings[train_idx], labels[train_idx])
        val_ds = EmbeddingDataset(embeddings[val_idx], labels[val_idx])
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        
        head = TransformerClassifier(in_features=embeddings.shape[1]).to(device)
        optimizer = torch.optim.AdamW(head.parameters(), lr=2e-3, weight_decay=1e-3)
        criterion = nn.CrossEntropyLoss(weight=weights)

        best_f1, best_acc = 0.0, 0.0
        for ep in range(1, epochs + 1):
            head.train()
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                loss = criterion(head(x), y)
                loss.backward()
                optimizer.step()
                
            head.eval()
            preds, targets = [], []
            with torch.no_grad():
                for x, y in val_loader:
                    preds.extend(head(x.to(device)).argmax(-1).cpu().numpy())
                    targets.extend(y.numpy())
                    
            val_f1 = f1_score(targets, preds, average="macro")
            val_acc = accuracy_score(targets, preds)
            if val_f1 > best_f1:
                best_f1, best_acc = val_f1, val_acc
                
        print(f"Fold {fold}/5 -> Macro-F1: {best_f1:.4f} | Accuracy: {best_acc * 100:.2f}%")
        fold_f1.append(best_f1)
        fold_acc.append(best_acc)
        
    print(f"\n===> MiniLM Mean Macro-F1 : {np.mean(fold_f1):.4f} (±{np.std(fold_f1):.4f})")
    print(f"===> MiniLM Mean Accuracy : {np.mean(fold_acc) * 100:.2f}%")
    return np.mean(fold_f1), np.mean(fold_acc), fold_f1, fold_acc

transformer_f1, transformer_acc, transformer_folds_f1, transformer_folds_acc = train_transformer_cv(embeddings_matrix, df["label"].values)

--- 
## 🇧🇩 Model 4: BUET BanglaBERT (`csebuetnlp/banglabert`)
Pretrained on 110 GB of Bengali text by CSE, BUET with a 32,000-token vocabulary and 768-dim embeddings.

In [ ]:
# 9. Extract BUET BanglaBERT Embeddings on GPU
BANGLABERT_ID = "csebuetnlp/banglabert"
print(f"Loading BUET BanglaBERT: {BANGLABERT_ID} ...")
bangla_tokenizer = AutoTokenizer.from_pretrained(BANGLABERT_ID)
bangla_model = AutoModel.from_pretrained(BANGLABERT_ID).to(device)
bangla_model.eval()

all_banglabert_embeddings = []
batch_size = 64
texts = df["review_text"].tolist()
print(f"Extracting BanglaBERT embeddings for {len(texts)} reviews on {device}...")

with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        enc = bangla_tokenizer(batch_texts, padding=True, truncation=True, max_length=96, return_tensors="pt").to(device)
        out = bangla_model(**enc)
        pooled = mean_pooling(out, enc["attention_mask"])
        normed = F.normalize(pooled, p=2, dim=1)
        all_banglabert_embeddings.append(normed.cpu().numpy())

banglabert_matrix = np.vstack(all_banglabert_embeddings)
print(f"Extracted BanglaBERT Embedding Matrix: {banglabert_matrix.shape}")

In [ ]:
# 10. Train & Evaluate BanglaBERT Head (5-Fold CV)
def train_banglabert_cv(embeddings, labels, epochs=15, batch_size=32):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_f1, fold_acc = [], []
    
    class_counts = pd.Series(labels).value_counts().sort_index().values
    weights = torch.tensor([len(labels) / (len(class_counts) * c) for c in class_counts], dtype=torch.float32).to(device)

    for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, labels), 1):
        train_ds = EmbeddingDataset(embeddings[train_idx], labels[train_idx])
        val_ds = EmbeddingDataset(embeddings[val_idx], labels[val_idx])
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        
        head = TransformerClassifier(in_features=768, hidden=256, dropout=0.3).to(device)
        optimizer = torch.optim.AdamW(head.parameters(), lr=2e-3, weight_decay=1e-3)
        criterion = nn.CrossEntropyLoss(weight=weights)

        best_f1, best_acc = 0.0, 0.0
        for ep in range(1, epochs + 1):
            head.train()
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                loss = criterion(head(x), y)
                loss.backward()
                optimizer.step()
                
            head.eval()
            preds, targets = [], []
            with torch.no_grad():
                for x, y in val_loader:
                    preds.extend(head(x.to(device)).argmax(-1).cpu().numpy())
                    targets.extend(y.numpy())
                    
            val_f1 = f1_score(targets, preds, average="macro")
            val_acc = accuracy_score(targets, preds)
            if val_f1 > best_f1:
                best_f1, best_acc = val_f1, val_acc
                
        print(f"Fold {fold}/5 -> Macro-F1: {best_f1:.4f} | Accuracy: {best_acc * 100:.2f}%")
        fold_f1.append(best_f1)
        fold_acc.append(best_acc)
        
    print(f"\n===> BanglaBERT Mean Macro-F1 : {np.mean(fold_f1):.4f} (±{np.std(fold_f1):.4f})")
    print(f"===> BanglaBERT Mean Accuracy : {np.mean(fold_acc) * 100:.2f}%")
    return np.mean(fold_f1), np.mean(fold_acc), fold_f1, fold_acc

banglabert_f1, banglabert_acc, banglabert_folds_f1, banglabert_folds_acc = train_banglabert_cv(banglabert_matrix, df["label"].values)

--- 
## 📊 Model Comparison & Visualizations

In [ ]:
# 11. Visual Benchmark Summary across All Paradigms
benchmark_df = pd.DataFrame({
    "Model": [
        "Dummy Baseline",
        "Char N-Gram Logistic Regression (ML)",
        "BiLSTM + Char-CNN (DL)",
        "Multilingual MiniLM (DL)",
        "BUET BanglaBERT (DL)",
        "Soft-Voting Ensemble (ML)"
    ],
    "Macro-F1": [0.2542, 0.7796, bilstm_f1, transformer_f1, banglabert_f1, ensemble_f1],
    "Accuracy (%)": [61.62, 85.96, bilstm_acc * 100, transformer_acc * 100, banglabert_acc * 100, ensemble_acc * 100]
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=benchmark_df, x="Macro-F1", y="Model", hue="Model", palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("5-Fold Stratified Macro-F1 Benchmark")
axes[0].set_xlim(0, 1.0)
for i, v in enumerate(benchmark_df["Macro-F1"]):
    axes[0].text(v + 0.02, i, f"{v:.4f}", va="center", fontweight="bold")

sns.barplot(data=benchmark_df, x="Accuracy (%)", y="Model", hue="Model", palette="magma", legend=False, ax=axes[1])
axes[1].set_title("Overall Classification Accuracy (%)")
axes[1].set_xlim(0, 100)
for i, v in enumerate(benchmark_df["Accuracy (%)"]):
    axes[1].text(v + 1.0, i, f"{v:.1f}%", va="center", fontweight="bold")

plt.tight_layout()
plt.show()
display(benchmark_df)

--- 
## 🧪 Interactive Real-Time Prediction
Test custom reviews in **Banglish**, **Bengali script**, or **English**:

In [ ]:
# 12. Train Final Production Models and Test Samples
# Fit final Ensemble ML
print("Fitting final Ensemble ML model...")
ensemble_pipeline.fit(df["review_text"], df["sentiment"])

# Fit final BiLSTM
print("Fitting final BiLSTM DL model...")
full_vocab = DualVocabulary()
full_vocab.fit(df["clean_text"].tolist())
final_bilstm = BiLSTMWithAttention(len(full_vocab.word2idx), len(full_vocab.char2idx)).to(device)
full_loader = DataLoader(BiLSTMAttentionDataset(df["clean_text"].tolist(), df["label"].tolist(), full_vocab), batch_size=32, shuffle=True)
class_counts = df["label"].value_counts().sort_index().values
weights = torch.tensor([len(df) / (len(class_counts) * c) for c in class_counts], dtype=torch.float32).to(device)
opt = torch.optim.AdamW(final_bilstm.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss(weight=weights)

for ep in range(1, 9):
    final_bilstm.train()
    for w_ids, c_ids, lbls in full_loader:
        w_ids, c_ids, lbls = w_ids.to(device), c_ids.to(device), lbls.to(device)
        opt.zero_grad()
        loss = crit(final_bilstm(w_ids, c_ids), lbls)
        loss.backward()
        opt.step()
print("All final models ready!")

def predict_all(text: str):
    print("=" * 60)
    print(f"Review: \"{text}\"")
    # 1. Ensemble ML Prediction
    ens_pred = ensemble_pipeline.predict([text])[0]
    ens_probs = ensemble_pipeline.predict_proba([text])[0]
    print(f"[Ensemble ML] -> {ens_pred:8s} | Probabilities: {dict(zip(ensemble_pipeline.classes_, [round(p, 4) for p in ens_probs]))}")
    
    # 2. BiLSTM DL Prediction
    final_bilstm.eval()
    w, c = full_vocab.encode(clean_text(text))
    w_t, c_t = torch.tensor([w], dtype=torch.long).to(device), torch.tensor([c], dtype=torch.long).to(device)
    with torch.no_grad():
        bilstm_probs = F.softmax(final_bilstm(w_t, c_t), dim=-1).squeeze(0).cpu().numpy()
    bilstm_pred = INV_LABEL_MAP[np.argmax(bilstm_probs)]
    print(f"[BiLSTM DL]   -> {bilstm_pred:8s} | Probabilities: {dict(zip(LABEL_MAP.keys(), [round(float(p), 4) for p in bilstm_probs]))}")
    print("=" * 60)

predict_all("Network speed khub e baje, 4G thakleo kono kaje ase na")
predict_all("খুব সুন্দর এবং সহজ অ্যাপ, অনেক সুবিধা পাই")
predict_all("মোটামুটি চলে তবে আরও উন্নতি দরকার")

--- 
## 💾 Save & Download Checkpoints and results.txt
Save trained models and export the full benchmark report to `results.txt`:

In [ ]:
# 13. Save Model Checkpoints & Export results.txt
import datetime
import joblib

os.makedirs("dl/models", exist_ok=True)
os.makedirs("ml/models", exist_ok=True)
os.makedirs("dl/reports", exist_ok=True)

# Save models
torch.save({
    "model_state_dict": final_bilstm.state_dict(),
    "word2idx": full_vocab.word2idx,
    "char2idx": full_vocab.char2idx,
    "label_map": LABEL_MAP
}, "dl/models/colab_bilstm_attention.pt")

joblib.dump(ensemble_pipeline, "ml/models/ensemble_voting_classifier.joblib")
print("Saved model artifacts to dl/models and ml/models!")

now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
hw_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

results_content = f"""================================================================================
TELECOM VOICE OF CUSTOMER (VoC) INTELLIGENCE - FULL PARADIGM BENCHMARK REPORT
================================================================================
Generated At  : {now_str}
Hardware Device: {hw_name}
Dataset Scope : 4,500 Multilingual Reviews (Grameenphone, Banglalink, Robi)

1. 5-FOLD STRATIFIED CROSS-VALIDATION SUMMARY:
--------------------------------------------------------------------------------
Model                                  Macro-F1 (Mean ± Std)   Accuracy (Mean ± Std)
--------------------------------------------------------------------------------
Statistical Floor (Dummy - Most Freq)  0.2542 ± 0.0001         61.62% ± 0.05%
Deep Learning (BiLSTM + Attention)     {bilstm_f1:.4f} ± {np.std(bilstm_folds_f1):.4f}         {bilstm_acc*100:.2f}% ± {np.std(bilstm_folds_acc)*100:.2f}%
Classical ML (Single LogReg)           0.7796 ± 0.0182         85.96% ± 1.36%
Deep Learning (MiniLM Transformer)     {transformer_f1:.4f} ± {np.std(transformer_folds_f1):.4f}         {transformer_acc*100:.2f}% ± {np.std(transformer_folds_acc)*100:.2f}%
Deep Learning (BUET BanglaBERT)        {banglabert_f1:.4f} ± {np.std(banglabert_folds_f1):.4f}         {banglabert_acc*100:.2f}% ± {np.std(banglabert_folds_acc)*100:.2f}%
Ensemble ML (Soft-Voting Classifier)   {ensemble_f1:.4f} ± {np.std(ensemble_folds_f1):.4f}         {ensemble_acc*100:.2f}% ± {np.std(ensemble_folds_acc)*100:.2f}%

2. SAVED ARTIFACTS:
--------------------------------------------------------------------------------
  - BiLSTM Model Checkpoint      : dl/models/colab_bilstm_attention.pt
  - Ensemble ML Model Artifact   : ml/models/ensemble_voting_classifier.joblib
  - Benchmark Summary Report     : results.txt
================================================================================
"""

with open("results.txt", "w", encoding="utf-8") as f:
    f.write(results_content)

with open("dl/reports/results.txt", "w", encoding="utf-8") as f:
    f.write(results_content)

print(results_content)
print(">> Saved benchmark report to results.txt!")

# from google.colab import files
# files.download("results.txt")
# files.download("ml/models/ensemble_voting_classifier.joblib")